# 02 — From stop visits to travel times

**Our question today:** which two stop visits can we safely use to calculate a travel time?

In Notebook 1, we explored a table. Today we will inspect possible data problems and work through **one bus on one day**. We will create *candidate* travel times, not a finished training dataset.

### Before starting

Place this notebook and `RTS_Route7_March2023.xlsx` in the **same folder**. Open that folder in VS Code and choose your Python kernel. Run cells in order with **Shift + Enter**.

Read a short section, run the cell, and pause to discuss the result. You do not need to understand all of pandas at once. We will keep the Excel file unchanged.

## 1. Load the libraries and data

`pandas` lets us work with tables. `display` lets us show more than one table in a cell. We reload the workbook so this notebook works independently of Notebook 1.

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.read_excel("RTS_Route7_March2023.xlsx", engine="openpyxl")
df.head()

## 2. Keep a working copy

We will add columns to `working`, leaving `df` unchanged. We also keep the original dataframe index as `source_row`, so we can trace examples back to the original table. Excel's row number is `source_row + 2`, because Excel starts at 1 and the first row contains headings.

In [ ]:
working = df.copy()
working["source_row"] = working.index

print("Original rows:", len(df))
working[["source_row", "Bus", "Date", "Stop ID", "Stop Seq ID", "Arrive"]].head()

## 3. Inspect missing values and repeated records

Before changing anything, ask what might be wrong. A missing value is not automatically zero. A repeated event key is not automatically an identical row.

In [ ]:
print("Missing values by column:")
display(df.isna().sum())

print("Extra exact copies across all original columns:", df.duplicated().sum())
event_columns = ["Date", "Bus", "Stop ID", "Arrive"]
print("Extra repeated event keys:", df.duplicated(subset=event_columns).sum())

repeated = df.duplicated(subset=event_columns, keep=False)
display(df.loc[repeated].head(6))

**Stop and discuss:** Do repeated event keys have different passenger or distance values? What could explain that? Would deleting them lose information?

**Our choice for this walkthrough:** delete nothing yet. We will flag ambiguity and inspect it. Later, the class can agree on a cleaning policy and explain it.

## 4. Combine the date and clock time

To subtract arrivals reliably, we need full timestamps. For example:

`2023-03-01` + `06:05:27` → `2023-03-01 06:05:27`

The first line reads calendar dates. The second reads clock times as time since midnight. Adding them creates `arrival_time`.

`errors="coerce"` marks values that cannot be parsed as missing, rather than guessing. We also reject clock values outside one day. We are assuming Date means calendar date; overnight records would need separate review.

In [ ]:
working["service_date"] = pd.to_datetime(working["Date"], errors="coerce").dt.normalize()
clock_time = pd.to_timedelta(working["Arrive"].astype(str), errors="coerce")
clock_time = clock_time.where((clock_time >= pd.Timedelta(0)) & (clock_time < pd.Timedelta(days=1)))
working["arrival_time"] = working["service_date"] + clock_time

print("Missing or invalid arrival timestamps:", working["arrival_time"].isna().sum())
working[["Date", "Arrive", "arrival_time"]].head()

## 5. Choose one bus and one day

We will use **bus 2105 on March 1, 2023** as our first example. These are real values in this workbook. You can change them later to investigate another bus-day.

`==` asks “is equal to?” and `&` means “and.” The selection keeps rows matching both conditions. A bus-day can still contain **many separate journeys**.

In [ ]:
BUS_TO_STUDY = 2105
DATE_TO_STUDY = "2023-03-01"

selected = (
    (working["Bus"] == BUS_TO_STUDY)
    & (working["service_date"] == pd.Timestamp(DATE_TO_STUDY))
)
bus_day = working.loc[selected].copy()

if bus_day.empty:
    raise ValueError("No matching rows. Choose a bus and date present in the workbook.")
if bus_day["arrival_time"].isna().any():
    raise ValueError("This bus-day has missing timestamps. Review those before ordering its events.")

bus_day = bus_day.sort_values("arrival_time", kind="stable").reset_index(drop=True)
print("Recorded visits for this bus-day:", len(bus_day))
bus_day[["source_row", "Stop ID", "Stop Name", "Stop Seq ID", "arrival_time"]].head(15)

**Notice:** the left-hand index now starts at 0 for this smaller table. `source_row` still points to the original table.

**Discuss:** does stop order keep increasing forever? What might a restart mean? If two visits share a timestamp, can sorting tell us which actually happened first?

## 6. Work through one example by hand

The first two displayed visits are *candidates* for A and B. `iloc[0]` selects the first row by position; `iloc[1]` selects the second. We will subtract their timestamps, but must still verify that they belong together.

In [ ]:
stop_a = bus_day.iloc[0]
stop_b = bus_day.iloc[1]

example_seconds = (stop_b["arrival_time"] - stop_a["arrival_time"]).total_seconds()
print("Stop A:", stop_a["Stop ID"], "at", stop_a["arrival_time"])
print("Stop B:", stop_b["Stop ID"], "at", stop_b["arrival_time"])
print("Candidate travel time:", example_seconds, "seconds")

For this starting example, you should see stops **1 → 292** and **86 seconds**.

Arithmetic alone does not prove this is a valid trip segment. Check that the stops are consecutive on the route and that there is no journey boundary between them.

## 7. Show the next recorded visit beside each current visit

Instead of manually repeating the subtraction, `shift(-1)` brings the following row's value onto the current row. Here we use it only inside our selected bus-day, not across the whole workbook.

**Important:** the next *recorded* visit is not necessarily the expected next stop. There may be missing visits, ties, or a new journey. We will flag these issues next.

In [ ]:
bus_day["next_stop"] = bus_day["Stop ID"].shift(-1)
bus_day["next_sequence"] = bus_day["Stop Seq ID"].shift(-1)
bus_day["next_arrival"] = bus_day["arrival_time"].shift(-1)
bus_day["next_source_row"] = bus_day["source_row"].shift(-1)
bus_day["travel_time_s"] = (
    bus_day["next_arrival"] - bus_day["arrival_time"]
).dt.total_seconds()

pair_columns = ["source_row", "Stop ID", "next_source_row", "next_stop",
                "arrival_time", "next_arrival", "travel_time_s"]
bus_day[pair_columns].head(12)

The last visit has no following row, so its next-arrival value and travel time are missing. That is expected; do not fill them with zero.

## 8. Flag examples that need a closer look

We will ask four simple questions:

1. Are both stop positions ordinary and consecutive (+1), with usable stop IDs?
2. Do either of the visits have an ambiguous timestamp shared with another record?
3. Is the difference positive?
4. Is it longer than our **provisional review threshold**?

`999` in stop order and `9999` in stop ID are unusual values in this dataset; keep them out of this first exercise until explained. The ten-minute threshold below is a discussion tool, not proof of bad data. A long interval could be a real delay. Change it only after discussing examples.

In [ ]:
REVIEW_THRESHOLD_SECONDS = 600

sequence = pd.to_numeric(bus_day["Stop Seq ID"], errors="coerce")
ordinary_visit = (
    sequence.notna() & (sequence >= 0) & (sequence % 1 == 0) & (sequence != 999)
    & bus_day["Stop ID"].notna() & (bus_day["Stop ID"] != 9999)
)
bus_day["flag_sequence_or_id"] = ~(
    ordinary_visit & ordinary_visit.shift(-1, fill_value=False)
    & ((bus_day["next_sequence"] - sequence) == 1)
)
tied_time = bus_day["arrival_time"].duplicated(keep=False)
bus_day["flag_tied_time"] = tied_time | tied_time.shift(-1, fill_value=False)
bus_day["flag_missing_or_nonpositive"] = (
    bus_day["travel_time_s"].isna() | (bus_day["travel_time_s"] <= 0)
)
bus_day["flag_long_interval"] = bus_day["travel_time_s"] > REVIEW_THRESHOLD_SECONDS

flag_columns = ["flag_sequence_or_id", "flag_tied_time",
                "flag_missing_or_nonpositive", "flag_long_interval"]
bus_day["passes_basic_checks"] = ~bus_day[flag_columns].any(axis=1)
bus_day[pair_columns + flag_columns].head(12)

### Look at the flagged examples

`~` means “not.” The next cell shows visits that **do not** pass the basic checks. Several flags can describe the same example, so their counts overlap.

**Discuss:** find a stop-order reset, a tied timestamp, or a long interval. What might each mean? Does changing the threshold solve the underlying uncertainty?

In [ ]:
display(bus_day[flag_columns].sum())
display(bus_day.loc[~bus_day["passes_basic_checks"], pair_columns + flag_columns].head(10))

## 9. Review five promising examples together

Passing the basic checks means **worth reviewing**, not automatically correct. This table takes up to five examples for classroom discussion. It is not a representative validation sample of the whole dataset.

For each example:

- Find both records and inspect the surrounding stops.
- Confirm expected A → B adjacency using a route pattern, historical schedule, or other documented evidence. The future row alone must not define the expected next stop.
- Check that both visits belong to the same journey, with no terminal boundary or missing stop.
- Calculate the seconds by hand.
- Record “accept,” “reject,” or “uncertain,” and explain why.

Leave uncertain cases out of any future approved dataset. Before modeling, repeat the review across different buses, dates, and route patterns.

In [ ]:
review_table = bus_day.loc[bus_day["passes_basic_checks"], pair_columns].head(5).copy()
review_table["decision"] = "not reviewed"
review_table["reason_or_evidence"] = ""
review_table

### Record the group's decisions

Edit the lists below in the same order as the displayed table. For example, write `"uncertain"` and explain the missing evidence. Do not write `"accept"` just because the code produced a positive number.

The default list lengths match our Route 7 example. If you select another bus-day with fewer than five candidates, shorten both lists.

In [ ]:
decisions = ["not reviewed", "not reviewed", "not reviewed", "not reviewed", "not reviewed"]
reasons = ["", "", "", "", ""]

if len(decisions) != len(review_table) or len(reasons) != len(review_table):
    raise ValueError("Use one decision and one reason for each row in review_table.")

review_table["decision"] = decisions
review_table["reason_or_evidence"] = reasons
review_table

## 10. Create features: what might help us predict?

A **target** is the answer we want to predict. A **feature** is information we give the model to help predict that answer.

Our target is already calculated: `travel_time_s`, from arrival at A to arrival at B. Now we will create a few possible inputs, one at a time. This is **feature engineering**.

| Column to create | Meaning | Why might it help? |
| --- | --- | --- |
| `stop_pair` | A label such as `7:1->292` | Different road segments take different amounts of time. |
| `hour` | Hour of arrival at A, from 0 to 23 | Conditions can change during the day. |
| `weekday` | Day name from arrival at A | Travel patterns may differ across days. |
| `time_band` | A simple group of hours | Makes time-of-day comparisons easier to read. |

Route and stop IDs are **labels**, not quantities. We include the route in the segment label and keep A → B different from B → A.

In [ ]:
bus_day["stop_pair"] = (
    "7:" + bus_day["Stop ID"].astype("Int64").astype(str)
    + "->" + bus_day["next_stop"].astype("Int64").astype(str)
)
bus_day["hour"] = bus_day["arrival_time"].dt.hour
bus_day["weekday"] = bus_day["arrival_time"].dt.day_name()

bus_day[["stop_pair", "arrival_time", "hour", "weekday", "travel_time_s"]].head(8)

`.dt.hour` extracts the hour from a timestamp. `.dt.day_name()` extracts its weekday.

**Discuss:** why do we extract the hour from arrival at A, rather than arrival at B? We are imagining a prediction made when the bus reaches A; arrival at B is still in the future.

The stop-pair label above describes the recorded candidate. Before using it in an operational prediction, independently confirm the route's expected next stop. Do not rely on the future row to reveal where the bus goes next.

### Group hours into readable time bands

`pd.cut` puts numbers into groups. Our example groups below are classroom choices, not official RTS peak periods. With `right=False`, 06:00 belongs to the 06–10 group and 10:00 belongs to the next group.

In [ ]:
bus_day["time_band"] = pd.cut(
    bus_day["hour"],
    bins=[0, 6, 10, 16, 19, 24],
    labels=["00–06", "06–10", "10–16", "16–19", "19–24"],
    right=False,
)
bus_day[["arrival_time", "hour", "time_band"]].head(8)

**Your turn:** explain which band includes 7 a.m., noon, and 5 p.m. Would you choose different groups? Explain why before changing them. Later we can compare hour versus time-band features using validation dates.

## 11. Add the previous segment time carefully

Imagine three verified consecutive stops: **P → A → B**.

- **Target:** arrival B minus arrival A — still unknown when the bus reaches A.
- **Previous segment feature:** arrival A minus arrival P — already known at A, if we have tracked that preceding segment.

A slow preceding segment might indicate the bus is already experiencing delays. It might also have little predictive value. We will test that later.

First, shift the candidate duration down one row with `shift(1)`, so the preceding interval appears beside the current interval. Only carry it forward when both neighboring candidate pairs pass our basic checks. This still produces a **candidate feature**, not a verified one.

In [ ]:
previous_passed = bus_day["passes_basic_checks"].shift(1, fill_value=False)
bus_day["candidate_previous_time_s"] = bus_day["travel_time_s"].shift(1).where(
    bus_day["passes_basic_checks"] & previous_passed
)

bus_day[["stop_pair", "candidate_previous_time_s", "travel_time_s"]].head(8)

The first example has no previous interval, so its value is missing. A new journey or an ambiguous preceding visit must also leave it missing. Do not fill it with the future target or carry it across journey boundaries.

### Keep manual approval separate

For our small review worksheet, a previous-segment feature is usable only when the class has accepted **both** the current and preceding pairs, with evidence that they form one continuous journey.

The next cell reads your decisions from Section 9. With all decisions still `not reviewed`, approved feature values remain missing. That is expected. After changing decisions, rerun the cells below.

In [ ]:
accepted_indices = review_table.index[
    review_table["decision"].astype(str).str.strip().str.lower().eq("accept")
    & review_table["reason_or_evidence"].astype(str).str.strip().ne("")
]
current_accepted = pd.Series(bus_day.index.isin(accepted_indices), index=bus_day.index)
previous_accepted = current_accepted.shift(1, fill_value=False)

bus_day["previous_segment_time_s"] = bus_day["candidate_previous_time_s"].where(
    current_accepted & previous_accepted
)

feature_columns = ["source_row", "stop_pair", "hour", "weekday", "time_band",
                   "previous_segment_time_s", "travel_time_s"]
feature_examples = bus_day.loc[bus_day["passes_basic_checks"], feature_columns].copy()
feature_examples.head(8)

This table is a **feature-preparation exercise**, not an approved training dataset. The current target and stop-pair identity still require the review described earlier. Accepting a row in this worksheet is your group's recorded judgment; the code cannot prove the evidence is correct.

## 12. Explore differences without fitting a model

Let's group the candidate examples by stop pair and time band. `groupby` collects matching rows; `agg` calculates a count and a median for each group.

We show counts because a median from one example gives much less evidence than a median from many. These exploratory medians are **not** our future prediction baseline: that baseline must be learned from training dates only.

In [ ]:
candidate_summary = feature_examples.groupby(
    ["stop_pair", "time_band"], observed=True
).agg(
    example_count=("travel_time_s", "size"),
    median_candidate_seconds=("travel_time_s", "median"),
).reset_index()

candidate_summary.head(15)

**Discuss:** could a segment have a different travel time at different hours? How many examples would you want before trusting that pattern?

We selected **one day**, so every row has the same weekday. We cannot learn weekday differences from this exercise. That requires verified examples across multiple dates. The same applies to claims about other buses or routes.

### The experiments these features prepare us for

| Later experiment | Inputs |
| --- | --- |
| Baseline | Training-period median for each directed stop pair. |
| Model A | Stop pair. |
| Model B | Stop pair + hour + weekday. |
| Model C | Model B + verified previous-segment time. |

Those models will be compared on the same later validation dates using error in seconds. Decide how to handle missing previous-segment values inside a training-only preprocessing pipeline. Do not let Model C appear better simply by scoring it on an easier, different set of rows.

**No model is trained here.** Notebook 3 establishes the baseline; Notebook 4 fits and compares models. Hour-versus-prediction plots will come after fitting, using combinations supported by the data.

## 13. Save your work and explain your decisions

Save the notebook first. The optional cell below can save your five-row **review worksheet** next to it. It is not a cleaned full dataset, and it does not prove the five rows are valid.

Leave the switch false until you want to create the file. Saving again overwrites the worksheet, so keep another copy if you want a record of earlier decisions.

In [ ]:
SAVE_REVIEW_WORKSHEET = False

if SAVE_REVIEW_WORKSHEET:
    review_table.to_csv("route7_stop_pair_review.csv", index=False)
    print("Saved route7_stop_pair_review.csv")
else:
    print("No file written. Save your notebook to keep today's work.")

## Discussion notes — edit this cell

1. **One flagged example and what might explain it:**
2. **A cleaning choice we agreed on, and why:**
3. **Evidence that two stops are consecutive on the route:**
4. **Why a bus-day is not the same thing as a journey:**
5. **What we still need to check before creating a full training table:**
6. **A feature we engineered and why it might help:**
7. **Why previous-segment time is different from the target:**
8. **Why this one-day example cannot establish weekday effects:**

### What should be finished after this walkthrough?

- You can explain how a travel-time target is calculated.
- You have inspected ambiguous records instead of deleting them blindly.
- You have up to five reviewed examples, with honest decisions and evidence.
- You have created stop-pair, hour, weekday, time-band, and carefully masked previous-segment features.
- You have written the questions your group must resolve next.

### What comes before Notebook 3?

With your instructor, apply the agreed rules to a clearly verified segment scope across multiple dates. Preserve original row references and review examples from other buses/dates. Then create an approved modeling table and a chronological train/validation/test split. This simplified notebook deliberately does **not** automate that whole process.

Reserve the final test dates before selecting thresholds or features. The default demonstration uses March 1, an early development date. If you change dates during cleaning discussions, stay within the agreed development period.

Notebook 3 will compare a simple historical-median prediction with later model results. Do not start fitting models just because a few arithmetic differences look reasonable.

**Data credit:** Richard Borzi, University of Florida; supplied by Gainesville RTS. [Route 7 dataset](https://zenodo.org/records/10034933), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).